[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/03%20Aprendizaje%20M%C3%A1quina/notebooks/03_Preparar_Datos_para_ML.ipynb)

# 03 · Preparar datos para ML

> **Pregunta guía:** ¿qué información estaría disponible antes de una hora de operación para anticipar la demanda de bicicletas?

Usaremos el conjunto real **Bike Sharing** de Capital Bikeshare, Washington, D. C. (2011–2012). Cada fila representa una hora. El objetivo es el total de alquileres, pero el dataset también incluye sus componentes: alquileres casuales y registrados.

**Al terminar:** definir X/y, inspeccionar una tabla real, identificar fuga de información, separar el futuro respetando el tiempo y aplicar preprocesamiento sólo con datos de entrenamiento.

📊 [Ficha UCI y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) · DOI 10.24432/C5W894.

## 1. Cargar el dataset

Los datos se descargan de UCI al ejecutar. No generamos observaciones ficticias ni guardamos una copia en el repositorio.

In [ ]:
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bike = fetch_ucirepo(id=275)
X_original = bike.data.features.copy()
y_original = bike.data.targets.copy()
df = X_original.copy()
df["cnt"] = y_original.iloc[:, 0].to_numpy()
df.head()

## 2. Entender la tabla antes de modelar

Primero preguntamos qué representa una fila y qué contiene cada columna. Los resúmenes ayudan a detectar tipos inesperados, faltantes, duplicados y rangos que merecen revisión.

In [ ]:
print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.dtypes.to_frame("tipo"))
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("faltantes"))
print("Filas duplicadas:", df.duplicated().sum())
display(df.describe(include="all").T)

### Visualización: ¿cómo se distribuye la demanda?

Un histograma muestra la forma de la variable objetivo. La figura sirve para hacer una pregunta sobre los datos; no basta con mostrarla y seguir.

In [ ]:
df["cnt"].hist(bins=40, color="#176b87", edgecolor="white")
plt.title("Alquileres de bicicletas por hora")
plt.xlabel("Alquileres (cnt)")
plt.ylabel("Número de horas")
plt.show()

## 3. Definir objetivo y reconocer leakage

El objetivo es **cnt**. **casual** y **registered** son componentes del total: incluirlos filtraría directamente la respuesta. **instant** identifica filas, no una característica causal. Usaremos la fecha para ordenar y separar periodos, no como predictor inicial.

En una aplicación, también debemos preguntar si el clima observado estaría disponible cuando se emite la predicción.

In [ ]:
target = "cnt"
leakage_and_id = [c for c in ["casual", "registered", "instant"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in [target, *leakage_and_id, "dteday"]]

print("Excluidas:", leakage_and_id)
print("Predictores candidatos:", feature_cols)
if {"casual", "registered", "cnt"}.issubset(df.columns):
    print("¿casual + registered = cnt en todas las filas?",
          (df["casual"] + df["registered"] == df["cnt"]).all())

## 4. Separar entrenamiento y prueba por fecha

Como la meta es anticipar demanda futura, reservamos el bloque de fechas más reciente para prueba. No mezclamos al azar horas de años distintos: eso podría dar una evaluación demasiado optimista.

In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"])
unique_dates = np.sort(df["dteday"].dropna().unique())
cutoff = unique_dates[int(0.80 * len(unique_dates))]

train = df[df["dteday"] < cutoff].copy()
test = df[df["dteday"] >= cutoff].copy()
X_train, y_train = train[feature_cols], train[target]
X_test, y_test = test[feature_cols], test[target]

print("Corte:", pd.Timestamp(cutoff).date())
print("Train:", X_train.shape, train["dteday"].min().date(), "→", train["dteday"].max().date())
print("Test:", X_test.shape, test["dteday"].min().date(), "→", test["dteday"].max().date())

## 5. Preprocesar dentro de un pipeline

Los códigos de estación, hora, día y clima representan categorías. El resto son valores numéricos. El imputador, escalador y codificador se ajustan con entrenamiento y luego se aplican a prueba sin volver a aprender sus parámetros.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
for col in ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]:
    if col in X_train.columns and col not in categorical:
        categorical.append(col)
numeric = [c for c in X_train.columns if c not in categorical]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
category_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", category_pipe, categorical)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Numéricas:", numeric)
print("Categóricas:", categorical)
print("Matrices:", X_train_ready.shape, X_test_ready.shape)

## 🧠 Qué aprendimos

- Una fila es una hora; X contiene predictores y y contiene la demanda total.
- Las columnas que componen el objetivo no son predictores válidos.
- Para pronosticar el futuro, la partición debe respetar el tiempo.
- Las transformaciones se ajustan dentro del conjunto de entrenamiento.

## 🧪 Mini reto

Mueve el corte temporal al 70 % y al 90 %. ¿Cómo cambia el tamaño del periodo de prueba? ¿Qué columnas excluirías si tu modelo tuviera que pronosticar demanda antes de conocer el clima real?

## 🚀 Siguiente

En la notebook 04 entrenaremos un modelo de regresión y compararemos sus errores con un baseline.

## Fuentes

- Fanaee-T, H. & Gama, J. (2013). UCI Bike Sharing Dataset. [Ficha, atribución y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI 10.24432/C5W894.
- Scikit-learn: [Pipelines y composición](https://scikit-learn.org/stable/modules/compose.html) y [errores comunes](https://scikit-learn.org/stable/common_pitfalls.html).